# Phase 1D.3-Lite Colab smoke
Run all cells on a Tesla T4 after providing the approved repository URL and immutable ref. This demo does not complete official Phase 1D.3.

In [ ]:
# Lite setup: provide the owner-authorized source URL and immutable commit/ref before Run All.
import os, subprocess, sys, json, re, time, getpass, platform, pathlib, shutil
REPO_URL = os.getenv("DACN_REPO_URL") or input("Owner-provided repository URL: ").strip()
REPO_REF = os.getenv("DACN_REPO_REF") or input("Owner-provided immutable commit/ref: ").strip()
if not REPO_URL or not REPO_REF:
    block("BLOCKED: source URL and commit/ref are required")
lite_evidence_dir = pathlib.Path("/content/lite-evidence")
def block(message):
    lite_evidence_dir.mkdir(parents=True, exist_ok=True)
    (lite_evidence_dir/"blocked.json").write_text(json.dumps({"status":"BLOCKED","diagnostic":str(message)}, indent=2)+"\n")
    raise RuntimeError(str(message))
repo = pathlib.Path("/content/dacn")
if repo.exists(): shutil.rmtree(repo)
try:
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REPO_REF], check=True)
except Exception as exc:
    block(f"BLOCKED: source checkout failed: {type(exc).__name__}")


In [ ]:
# Verify the T4 and preserve Colab's existing Torch installation before any package install.
import subprocess, json, sys, pathlib, platform
if platform.machine() not in ("x86_64", "amd64"):
    block("BLOCKED: Lite Node install supports Colab Linux x64 only")
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,uuid,memory.total", "--format=csv,noheader"], text=True).strip().splitlines()
if len(gpu) != 1 or "T4" not in gpu[0]:
    block(f"BLOCKED: expected one Tesla T4, found {gpu}")
gpu_name, gpu_uuid, gpu_memory = [item.strip() for item in gpu[0].split(",", 2)]
torch_before = subprocess.check_output([sys.executable, "-c", "import torch; print(torch.__version__); print(torch.__file__); print(torch.cuda.is_available())"], text=True).strip().splitlines()
if len(torch_before) != 3 or torch_before[2] != "True":
    block("BLOCKED: existing Colab Torch/CUDA unavailable")
constraint = pathlib.Path("/content/lite-torch-constraint.txt")
constraint.write_text(f"torch=={torch_before[0]}\n")
report = pathlib.Path("/content/lite-pip-report.json")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run", "--report", str(report), "-r", str(repo/"ai/requirements-lite.txt"), "-c", str(constraint)], check=True)
    subprocess.run([sys.executable, "-c", "import sys; sys.path.insert(0, str(__import__('pathlib').Path('/content/dacn'))); from ai.benchmark.lite_smoke import validate_pip_report; validate_pip_report('/content/lite-pip-report.json')"], check=True)
except Exception as exc:
    block(f"BLOCKED: dry-run report missing, malformed, or unsafe: {type(exc).__name__}")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo/"ai/requirements-lite.txt"), "-c", str(constraint)], check=True)
torch_after = subprocess.check_output([sys.executable, "-c", "import torch; print(torch.__version__); print(torch.__file__); print(torch.cuda.is_available())"], text=True).strip().splitlines()
if torch_after != torch_before:
    block("BLOCKED: existing Torch changed")


In [ ]:
# Install supported Node/pnpm and only the API workspace dependencies.
import subprocess, shutil, re, pathlib, os, platform
node = shutil.which("node")
node_ok = False
if node:
    version = subprocess.check_output([node, "--version"], text=True).strip().lstrip("v").split(".")
    node_ok = int(version[0]) == 24 and (int(version[1]), int(version[2])) >= (18, 0)
if not node_ok:
    archive = "/content/node-v24.18.0-linux-x64.tar.xz"
    subprocess.run(["curl", "-fsSL", "https://nodejs.org/dist/v24.18.0/node-v24.18.0-linux-x64.tar.xz", "-o", archive], check=True)
    subprocess.run(["tar", "-xJf", archive, "-C", "/content"], check=True)
    os.environ["PATH"] = "/content/node-v24.18.0-linux-x64/bin:" + os.environ["PATH"]
subprocess.run(["corepack", "prepare", "pnpm@10.12.1", "--activate"], cwd=repo, check=True)
subprocess.run(["pnpm", "install", "--frozen-lockfile", "--filter", "@dacn/api..."], cwd=repo, check=True)


In [ ]:
# Start the actual FastAPI worker once; keep the token hidden and wait for lifespan READY.
import subprocess, os, pathlib, time, re, getpass, json
runtime_token = getpass.getpass("AI runtime token (hidden): ")
if not runtime_token:
    block("BLOCKED: runtime token is required")
env = os.environ.copy()
env.update({"AI_RUNTIME_TOKEN": runtime_token, "AI_RUNTIME_GPU_UUID": gpu_uuid,
            "AI_RUNTIME_MODEL_ID": "Qwen/Qwen2.5-1.5B-Instruct",
            "AI_RUNTIME_MODEL_REVISION": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
            "AI_RUNTIME_LOAD_MODE": "fp16"})
log = open("/content/lite-uvicorn.log", "w")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "ai.runtime.app:app", "--host", "127.0.0.1", "--port", "8000"], cwd=repo, env=env, stdout=log, stderr=subprocess.STDOUT)
deadline = time.time() + 600
while time.time() < deadline:
    log.flush()
    contents = pathlib.Path("/content/lite-uvicorn.log").read_text(errors="replace")
    if "Application startup complete" in contents:
        break
    if server.poll() is not None:
        block("BLOCKED: Uvicorn startup failed; see /content/lite-uvicorn.log")
    time.sleep(2)
else:
    server.terminate()
    block("BLOCKED: worker did not reach lifespan READY")
match = re.findall(r"AI model loaded in ([0-9.]+) seconds", contents)
if not match:
    block("BLOCKED: model-load duration missing from worker startup log")
model_load_seconds = float(match[-1])


In [ ]:
# Run six actual outputs, then the opt-in real NestJS → loopback FastAPI request.
import subprocess, pathlib, json, os, importlib.metadata as metadata
outdir = pathlib.Path("/content/lite-evidence")
outdir.mkdir(exist_ok=True)
env.update({"AI_RUNTIME_BASE_URL": "http://127.0.0.1:8000", "AI_RUNTIME_TOKEN": runtime_token,
            "AI_RUNTIME_MODEL_ID": "Qwen/Qwen2.5-1.5B-Instruct", "AI_RUNTIME_MODEL_REVISION": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"})
evidence = {"modelId": env["AI_RUNTIME_MODEL_ID"], "modelRevision": env["AI_RUNTIME_MODEL_REVISION"],
            "gpu": gpu_name, "gpuUuid": gpu_uuid, "gpuMemory": gpu_memory,
            "torchVersion": torch_after[0], "torchPath": torch_after[1], "modelLoadSeconds": model_load_seconds,
            "dependencyVersions": {name: metadata.version(name) for name in ("fastapi", "uvicorn", "jsonschema", "transformers", "accelerate")}}
env.update({"AI_RUNTIME_GPU_NAME": gpu_name, "AI_RUNTIME_GPU_MEMORY": gpu_memory,
            "AI_RUNTIME_TORCH_VERSION": torch_after[0], "AI_RUNTIME_TORCH_PATH": torch_after[1],
            "AI_RUNTIME_MODEL_LOAD_SECONDS": str(model_load_seconds)})
fixtures = [json.loads(line) for line in (repo/"ai/benchmark/fixtures/lite-smoke.jsonl").read_text().splitlines() if line.strip()]
subprocess.run(["pnpm", "--filter", "@dacn/api", "exec", "tsx", "../../ai/benchmark/validate-inputs.ts"], cwd=repo, input=json.dumps(fixtures, ensure_ascii=False), text=True, check=True)
run = subprocess.run([sys.executable, "-m", "ai.benchmark.lite_smoke", "--output-dir", str(outdir)], cwd=repo, env=env, text=True, capture_output=True)
print(run.stdout)
if run.returncode:
    print(run.stderr.replace(runtime_token, "[REDACTED]"))
    raise RuntimeError("Lite smoke failed; preserve /content/lite-evidence")
api_env = env.copy()
api_env["AI_RUNTIME_LIVE_SMOKE"] = "1"
api_env["AI_RUNTIME_BASE_URL"] = "http://127.0.0.1:8000"
subprocess.run(["pnpm", "--filter", "@dacn/api", "exec", "jest", "--runInBand", "src/ai/ai-runtime-wire.spec.ts", "-t", "live Colab runtime"], cwd=repo, env=api_env, check=True)
pathlib.Path(outdir/"colab-evidence.json").write_text(json.dumps(evidence, indent=2)+"\n")
print("Evidence saved at", outdir)
